## 5) Structured Output

Usually, AI likes to chat and write paragraphs. But sometimes, you want a form filled out, not a conversation.
If you ask for a date range, you don't want *"Sure! The start date is..."*. You want `{"start": "2025-01-01", "end": "2025-01-05"}`.
Structured output forces the AI to stop chatting and generate data matching a specific schema.

###  Code Context
In the code below:
* `class DateRange(BaseModel)`: We define the "Form" we want the AI to fill out using Pydantic.
* `llm.with_structured_output(DateRange)`: We tell the LLM, "Your output **must** match this class."
* The result is an actual Python object, not a string.

📚 **[Docs: Structured Output](https://docs.langchain.com/oss/python/langchain/models#structured-output)**

![Diagram](./images/05_img.png)

# Supports
    # - pydantic
    # - Typedict
    # - Dataclass 
    # - Json schema

In [2]:
import os
import sys
from pathlib import Path

print(sys.executable)
project_root = Path.cwd()
while project_root != project_root.parent and not (
    project_root / "utils" / "env_utils.py"
).is_file():
    project_root = project_root.parent

if not (project_root / "utils" / "env_utils.py").is_file():
    raise FileNotFoundError("Could not locate utils/env_utils.py from the notebook directory.")

os.chdir(project_root)
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
from utils.env_utils import doublecheck_env, doublecheck_pkgs

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")  # check environmental variables
doublecheck_pkgs(pyproject_path="pyproject.toml", verbose=True)   # check packages

/Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week1-langchain/.venv/bin/python
Did not find file example.env.
This is used to double check the key settings for the notebook.
This is just a check and is not required.

Python 3.13.5 satisfies requires-python: >=3.11, <3.14
package                | required | installed | status | path                                                                            
---------------------- | -------- | --------- | ------ | --------------------------------------------------------------------------------
jupyter                | >=1.0.0  | 1.1.1     | ✅ OK   | …ai-certification-gen-academy/week1-langchain/.venv/lib/python3.13/site-packages
langchain              | >=1.3.10 | 1.4.5     | ✅ OK   | …ai-certification-gen-academy/week1-langchain/.venv/lib/python3.13/site-packages
langchain-anthropic    | >=1.4.6  | 1.7.6     | ✅ OK   | …ai-certification-gen-academy/week1-langchain/.venv/lib/python3.13/site-packag

In [6]:
from typing import TypedDict
from pydantic import BaseModel
from langchain_prompts import DATE_EXTRACTOR_PROMPT_TEMPLATE
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from datetime import datetime



# # You can define structure with Pydantic
# class DateRange(BaseModel):
#     start_date: str = Field(default = None, description="Start Date of the date range")
#     end_date: str = Field(default = None, description="End Date of the date range")

# OR you can define structure with TypedDict
class DateRange(TypedDict):
    start_date: str
    end_date: str

llm = init_chat_model(model="gpt-4o-mini", temperature=0)

structured_llm = llm.with_structured_output(DateRange)

prompt = ChatPromptTemplate.from_messages([
    ("system", DATE_EXTRACTOR_PROMPT_TEMPLATE),
    ("human", "{user_query}")
])


user_query = "I need time off for the next 4 days"
todays_date = datetime.now().strftime("%Y-%m-%d")


chain = prompt | structured_llm

llm_response = chain.invoke({'todays_date': todays_date, 'user_query': user_query})

llm_response

{'start_date': '2026-10-14', 'end_date': '2026-10-17'}